# Stage 09 - Answer Eval, Cache, Batch

C0: environment bootstrap and ground truth validation.

In [2]:
from pathlib import Path
import json

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
GT_PATH = ROOT / "data" / "ground_truth.json"
payload = json.loads(GT_PATH.read_text(encoding="utf-8"))
print("version:", payload.get("version"))
print("query_count:", len(payload.get("queries", [])))
for i, item in enumerate(payload.get("queries", []), start=1):
    print(f"{i}.", item["query"], "| key_phrases:", len(item.get("key_phrases", [])))

version: stage09-v1
query_count: 4
1. What is the treatment for MI? | key_phrases: 7
2. metformin cardiovascular effects | key_phrases: 6
3. papers on malaria after 2015 | key_phrases: 6
4. warfarin atrial fibrillation elderly | key_phrases: 7


In [3]:
# C0.5: dependency check (run once per kernel)
import importlib.util
import subprocess
import sys

required = ["rouge_score"]
missing = [m for m in required if importlib.util.find_spec(m) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rouge-score"])
else:
    print("Dependencies OK:", required)

Dependencies OK: ['rouge_score']


## C1：AnswerEvaluator 单条演示

> 幻觉分是**风险信号**，不是最终真伪裁定。该分数用于筛查可疑绝对化表述，需结合引用与人工复核解读。

In [4]:
import sys
from pathlib import Path

# Robust path bootstrap for notebook execution in either:
# - stage09 root as cwd, or
# - stage09/notebooks as cwd
CWD = Path.cwd().resolve()
STAGE09_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SRC = STAGE09_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bootstrap import bootstrap_paths

_ = bootstrap_paths(STAGE09_ROOT)

from answer_evaluator import AnswerEvaluator

evaluator = AnswerEvaluator()

sample_generated = (
    "Warfarin can reduce stroke risk in elderly atrial fibrillation patients, "
    "but bleeding risk remains and INR monitoring is required."
)
sample_reference = (
    "In elderly AF patients, warfarin reduces stroke risk with careful INR "
    "monitoring and bleeding risk assessment."
)
sample_gt_phrases = [
    "warfarin",
    "stroke risk",
    "INR monitoring",
    "bleeding risk",
]

result = evaluator.evaluate(
    generated=sample_generated,
    reference=sample_reference,
    gt_key_phrases=sample_gt_phrases,
)
result.to_dict()

{'rouge': {'rouge1': 0.6857, 'rouge2': 0.303, 'rougeL': 0.3429},
 'key_info_recall': 1.0,
 'key_info_matched': ['warfarin',
  'stroke risk',
  'inr monitoring',
  'bleeding risk'],
 'key_info_missing': [],
 'hallucination_risk': 0.0,
 'hallucination_signals': [],
 'readability': {'num_sentences': 1.0,
  'num_words': 19.0,
  'avg_sentence_len_words': 19.0,
  'avg_word_len_chars': 5.7895}}

## C2：缓存 miss -> hit 演示

同一组输入首次查询为 miss，写入后再次查询应命中（hit）。

In [5]:
from generation_cache import GenerationCache

cache = GenerationCache(max_entries=4, ttl_seconds=300, max_temperature=0.3)

query = "warfarin atrial fibrillation elderly"
context_text = "warfarin ... INR monitoring ... bleeding risk"
model = "deepseek-r1:7b"
temperature = 0.2

key = cache.make_key(query, context_text, model, temperature)

first = cache.get(key)  # miss
cache.set(key, {"answer": "cached answer demo", "sources": ["PMC123"]}, temperature=temperature)
second = cache.get(key)  # hit

{
    "first_get": first,
    "second_get": second,
    "stats": cache.stats(),
}

{'first_get': None,
 'second_get': {'answer': 'cached answer demo', 'sources': ['PMC123']},
 'stats': {'hits': 1, 'misses': 1, 'evictions': 0, 'size': 1}}

## C3：TTL 过期与高温不缓存演示

- TTL 到期后应返回 `None`
- 温度高于阈值时 `set()` 应返回 `False`

In [6]:
class _FakeClock:
    def __init__(self, start=1000.0):
        self.t = float(start)

    def now(self):
        return self.t

    def advance(self, sec):
        self.t += float(sec)


clock = _FakeClock()
cache_ttl = GenerationCache(ttl_seconds=5, now_fn=clock.now)
key_ttl = cache_ttl.make_key("q", "ctx", "m", 0.2)
cache_ttl.set(key_ttl, {"answer": "ttl demo"}, temperature=0.2)

before_expire = cache_ttl.get(key_ttl)
clock.advance(6)
after_expire = cache_ttl.get(key_ttl)

cache_temp = GenerationCache(max_temperature=0.3)
key_hot = cache_temp.make_key("q-hot", "ctx", "m", 0.9)
hot_set_ok = cache_temp.set(key_hot, {"answer": "should not cache"}, temperature=0.9)

{
    "ttl_before_expire": before_expire,
    "ttl_after_expire": after_expire,
    "high_temp_set_ok": hot_set_ok,
    "high_temp_get": cache_temp.get(key_hot),
}

{'ttl_before_expire': {'answer': 'ttl demo'},
 'ttl_after_expire': None,
 'high_temp_set_ok': False,
 'high_temp_get': None}

## C4：批量并行 + 失败隔离演示

并行发生在多 query 之间；单 query 内部流程仍保持串行。

In [7]:
import time
from batch_runner import BatchRunner

runner = BatchRunner(max_workers=3)
queries = [
    "What is the treatment for MI?",
    "bad-query-demo",
    "metformin cardiovascular effects",
    "warfarin atrial fibrillation elderly",
]


def mock_task(query: str) -> dict:
    # Simulate different runtimes and one failure.
    if "metformin" in query:
        time.sleep(0.2)
    elif "warfarin" in query:
        time.sleep(0.1)
    if query == "bad-query-demo":
        raise ValueError("simulated task failure")
    return {"answer_preview": query[:24]}

results = runner.run_batch(queries, mock_task)
stats = runner.summarize(results).to_dict()

{"results": results, "stats": stats}

{'results': [{'_index': 0,
   'query': 'What is the treatment for MI?',
   'status': 'ok',
   'latency_seconds': 0.0,
   'answer_preview': 'What is the treatment fo'},
  {'_index': 1,
   'query': 'bad-query-demo',
   'status': 'error',
   'latency_seconds': 0.0,
   'error': 'simulated task failure'},
  {'_index': 2,
   'query': 'metformin cardiovascular effects',
   'status': 'ok',
   'latency_seconds': 0.2007,
   'answer_preview': 'metformin cardiovascular'},
  {'_index': 3,
   'query': 'warfarin atrial fibrillation elderly',
   'status': 'ok',
   'latency_seconds': 0.1004,
   'answer_preview': 'warfarin atrial fibrilla'}],
 'stats': {'total': 4,
  'succeeded': 3,
  'failed': 1,
  'avg_latency_seconds': 0.0753,
  'error_rate': 0.25}}

## C5：端到端粘合演示（generation + cache + evaluation）

本单元演示 `PipelineWithEval.run_with_cache_and_eval()` 的统一输出结构。

In [8]:
from model_adapter import GenerationRequest, GenerationResponse
from pipeline_with_eval import PipelineWithEval


class _NotebookFakeAdapter:
    def __init__(self):
        self.calls = 0

    def generate(self, request: GenerationRequest) -> GenerationResponse:
        self.calls += 1
        ans = f"Mock answer for: {request.query}"
        return GenerationResponse(
            answer=ans,
            sources=[{"doc_id": "PMC-DEMO", "source_title": "demo"}],
            model_name=request.model_name or "mock-model",
            provider="mock",
            raw={"answer": ans, "sources": [{"doc_id": "PMC-DEMO", "source_title": "demo"}]},
        )


fake_adapter = _NotebookFakeAdapter()
pipe_eval = PipelineWithEval(
    model_adapter=fake_adapter,
    evaluator=AnswerEvaluator(),
    cache=GenerationCache(ttl_seconds=120),
    provider="mock",
    default_model_name="mock-model",
)

first = pipe_eval.run_with_cache_and_eval(
    query="warfarin atrial fibrillation elderly",
    ground_truth_entry={
        "reference_answer": "Warfarin in elderly AF requires INR monitoring and bleeding risk assessment.",
        "key_phrases": ["warfarin", "INR monitoring", "bleeding risk"],
    },
    context_text="warfarin ... INR monitoring ... bleeding risk",
    temperature=0.2,
)

second = pipe_eval.run_with_cache_and_eval(
    query="warfarin atrial fibrillation elderly",
    ground_truth_entry={
        "reference_answer": "Warfarin in elderly AF requires INR monitoring and bleeding risk assessment.",
        "key_phrases": ["warfarin", "INR monitoring", "bleeding risk"],
    },
    context_text="warfarin ... INR monitoring ... bleeding risk",
    temperature=0.2,
)

{
    "first_cache_hit": first["cache"]["hit"],
    "second_cache_hit": second["cache"]["hit"],
    "adapter_calls": fake_adapter.calls,
    "evaluation_preview": second["evaluation"],
}

{'first_cache_hit': False,
 'second_cache_hit': True,
 'adapter_calls': 1,
 'evaluation_preview': {'rouge': {'rouge1': 0.2222,
   'rouge2': 0.0,
   'rougeL': 0.2222},
  'key_info_recall': 0.3333,
  'key_info_matched': ['warfarin'],
  'key_info_missing': ['inr monitoring', 'bleeding risk'],
  'hallucination_risk': 0.0,
  'hallucination_signals': [],
  'readability': {'num_sentences': 1.0,
   'num_words': 7.0,
   'avg_sentence_len_words': 7.0,
   'avg_word_len_chars': 6.5714}}}

## C6：汇总导出与关键指标结论

导出 `outputs/samples/eval_cache_batch_report.json`，并展示命中率、平均耗时、评估分数分布、失败样本。

In [9]:
import json
import time
from pathlib import Path

from batch_runner import BatchRunner
from model_adapter import GenerationRequest, GenerationResponse, SnapshotModelAdapter
from pipeline_with_eval import PipelineWithEval
from report_builder import build_eval_cache_batch_report

STAGE09_ROOT = ROOT
GT_BY_QUERY = {item["query"]: item for item in payload["queries"]}

snapshot_path = STAGE09_ROOT.parent / "08 生成模块与提示词工程第二部分" / "outputs" / "samples" / "generation_eval.json"
snapshot_payload = json.loads(snapshot_path.read_text(encoding="utf-8"))
snapshots = {item["query"]: item for item in snapshot_payload.get("queries", [])}
model_name = snapshot_payload.get("config", {}).get("model", "snapshot-model")

pipe_export = PipelineWithEval(
    model_adapter=SnapshotModelAdapter(snapshots, provider="stage08_snapshot", model_name=model_name),
    evaluator=AnswerEvaluator(),
    cache=GenerationCache(ttl_seconds=600),
    provider="stage08_snapshot",
    default_model_name=model_name,
)

queries = [item["query"] for item in payload["queries"]]


def _task(query: str) -> dict:
    started = time.perf_counter()
    result = pipe_export.run_with_cache_and_eval(
        query,
        ground_truth_entry=GT_BY_QUERY[query],
        context_text=f"ctx::{query}",
        temperature=0.2,
        model_name=model_name,
    )
    result["latency_seconds"] = round(time.perf_counter() - started, 4)
    result["status"] = "ok"
    return result


runner = BatchRunner(max_workers=2)
first_pass = runner.run_batch(queries, _task)
second_pass = runner.run_batch(queries, _task)
batch_stats = runner.summarize(second_pass).to_dict()

report = build_eval_cache_batch_report(
    mode="offline",
    config={"temperature": 0.2, "max_workers": 2, "model_name": model_name},
    first_pass=first_pass,
    second_pass=second_pass,
    batch_stats=batch_stats,
)

out_path = STAGE09_ROOT / "outputs" / "samples" / "eval_cache_batch_report.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

summary = report["summary"]
conclusion = {
    "cache_hit_rate_first": summary["cache_first_pass"]["hit_rate"],
    "cache_hit_rate_second": summary["cache_second_pass"]["hit_rate"],
    "avg_latency_seconds": summary["avg_latency_seconds"],
    "rouge1_avg": summary["evaluation_first_pass"]["rouge1_avg"],
    "key_info_recall_avg": summary["evaluation_first_pass"]["key_info_recall_avg"],
    "hallucination_risk_avg": summary["evaluation_first_pass"]["hallucination_risk_avg"],
    "failures_first_pass": summary["failures_first_pass"],
    "failures_second_pass": summary["failures_second_pass"],
    "saved_to": str(out_path),
}
conclusion

{'cache_hit_rate_first': 0.0,
 'cache_hit_rate_second': 1.0,
 'avg_latency_seconds': 0.0041,
 'rouge1_avg': 0.0768,
 'key_info_recall_avg': 0.2321,
 'hallucination_risk_avg': 0.0,
 'failures_first_pass': [],
 'failures_second_pass': [],
 'saved_to': 'D:\\谷歌\\09 生成答案评估，缓存策略与批量处理\\outputs\\samples\\eval_cache_batch_report.json'}